In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
from collections import defaultdict
from PIL import Image
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report

import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input, Conv2D, GlobalAveragePooling2D, Dense, 
    BatchNormalization, Activation, Dropout
)
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2

SEED = 4242
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

img_dir = '../flipped_images'
excel_path = '../tabular.xlsx'

variant_keys = ['west_original', 'east_flip_horizontal', 'north_flip_vertical', 'south_flip_both']
image_names_to_select = ['AC QPS (clear)', 'Rest QGS (clear)', 'Stress QGS (clear)']
target_labels = ['normal', 'ischemia', 'infarction']

def pad_image(img, target_size=(224, 224)):
    width, height = img.size
    new_width, new_height = target_size
    if width > new_width or height > new_height:
        img.thumbnail((new_width, new_height), Image.LANCZOS)
        width, height = img.size 
    padding_width = (new_width - width) // 2
    padding_height = (new_height - height) // 2
    return np.pad(np.array(img),
                  ((padding_height, new_height - height - padding_height),
                   (padding_width, new_width - width - padding_width),
                   (0, 0)),
                  mode='constant', constant_values=0)

def plot_and_save_history(history, title, filename):
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(filename)
    plt.close()
    print(f"Saved {title} training plot to {filename}")

data = pd.read_excel(excel_path)
label_map = {'Normal': 0, 'Ischemia': 1, 'Infarction': 2}
data['Label'] = data['Label'].map(label_map)
data = pd.get_dummies(data, columns=['Sex'])

features = data.drop(['Label', 'ID'], axis=1)
correlation = features.corr().abs()
upper = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] == 1)]
data = data.drop(columns=to_drop)

tab_cols = [c for c in data.columns if c not in ['ID', 'Label']]
patient_data_dict = {}
for _, row in data.iterrows():
    patient_data_dict[int(row['ID'])] = {
        'features': row[tab_cols].values.astype(float),
        'label': int(row['Label'])
    }

X_img, X_tab, y_matched = [], [], []

for folder in os.listdir(img_dir):
    folder_path = os.path.join(img_dir, folder)
    if not os.path.isdir(folder_path): continue

    label_idx = next((target_labels.index(lbl) for lbl in target_labels if lbl in folder.lower()), None)
    if label_idx is None: continue

    files = [f for f in os.listdir(folder_path) if f.startswith('cropped')]
    grouped = defaultdict(lambda: defaultdict(dict))
    
    for f in files:
        match = re.match(r'cropped_(\d+)\s+(.+?)_(.+?)\.jpg$', f) 
        if match:
            pat_id, view_name, variant_key = match.groups()
            pat_id = int(pat_id)
            if view_name in image_names_to_select and variant_key in variant_keys:
                grouped[pat_id][variant_key][view_name] = f

    for pat_id, variants in grouped.items():
        if pat_id not in patient_data_dict: continue
            
        pat_tabular = patient_data_dict[pat_id]['features']
        pat_label = patient_data_dict[pat_id]['label']
        
        for variant_key in variant_keys:
            views = variants.get(variant_key, {})
            patient_images = []
            
            for view in image_names_to_select:
                matched_file = views.get(view)
                if matched_file:
                    img_path = os.path.join(folder_path, matched_file)
                    img = Image.open(img_path).convert('RGB')
                    padded_img = pad_image(img, target_size=(224, 224))
                    img_arr = preprocess_input(np.array(padded_img))
                else:
                    img_arr = preprocess_input(np.zeros((224, 224, 3), dtype=np.float32))
                patient_images.append(img_arr)
                
            stacked = np.concatenate(patient_images, axis=-1)
            X_img.append(stacked)
            X_tab.append(pat_tabular)
            y_matched.append(pat_label)

X_img = np.array(X_img)
X_tab = np.array(X_tab)
y_cat = to_categorical(np.array(y_matched), num_classes=3)

print(f"Data Aligned. Total combined samples: {X_img.shape[0]}")


X_train_img, X_val_img, X_train_tab, X_test_tab, y_train_cat, y_test_cat = train_test_split(
    X_img, X_tab, y_cat, test_size=0.2, random_state=42, stratify=np.argmax(y_cat, axis=1)
)

raw_mean_vals = np.nanmean(X_train_tab, axis=0)
X_train_tab = np.nan_to_num(X_train_tab, nan=raw_mean_vals)
X_test_tab = np.nan_to_num(X_test_tab, nan=raw_mean_vals) 

# ANOVA
selector = SelectKBest(score_func=f_classif, k=20)
X_train_sel = selector.fit_transform(X_train_tab, np.argmax(y_train_cat, axis=1))
X_test_sel = selector.transform(X_test_tab)


selected_mask = selector.get_support()
print("\nTop 20 Features selected by ANOVA:")
print(selected_features)

scaler = MinMaxScaler()
X_train_sel = scaler.fit_transform(X_train_sel)
X_test_sel = scaler.transform(X_test_sel)

scaled_selected_means = np.nanmean(X_train_sel, axis=0), # the scaled means of just the 20 features to save for the UI

input_9ch = Input(shape=(224, 224, 9), name="input_9channel")
reduced = Conv2D(3, (1, 1), activation='relu')(input_9ch)
resnet_base = ResNet50(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
resnet_base.trainable = False 
for layer in resnet_base.layers[-10:]: layer.trainable = True

x = resnet_base(reduced)
x = GlobalAveragePooling2D()(x)
x = Dense(64, kernel_regularizer=l2(0.005))(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.7)(x)
output = Dense(3, activation='softmax', kernel_regularizer=l2(0.005))(x)

cnn_model = Model(inputs=input_9ch, outputs=output)
cnn_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002), 
                  loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.005), 
                  metrics=['accuracy'])

cnn_history = cnn_model.fit(X_train_img, y_train_cat, validation_data=(X_val_img, y_test_cat), 
              epochs=50, batch_size=32, callbacks=[ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)])

plot_and_save_history(cnn_history, "CNN Model", "cnn_training_history.png")

y_train_labels_flat = np.argmax(y_train_cat, axis=1)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_labels_flat), y=y_train_labels_flat)
class_weight_dict = {0: class_weights[0], 1: class_weights[1], 2: class_weights[2]}

ann_model = Sequential([
    Input(shape=(20,)),
    Dense(64, activation='relu'),            
    Dense(64, activation='relu'),             
    Dense(3, activation='softmax')           
])
ann_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.002), loss='categorical_crossentropy', metrics=['accuracy'])

ann_history = ann_model.fit(X_train_sel, y_train_cat, epochs=65, batch_size=32, validation_data=(X_test_sel, y_test_cat), 
              callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)], class_weight=class_weight_dict)

plot_and_save_history(ann_history, "ANN Model", "ann_training_history.png")

cnn_pred = cnn_model.predict(X_val_img)
ann_pred = ann_model.predict(X_test_sel)

fused_pred = (0.6 * cnn_pred) + (0.4 * ann_pred)
final_classes = np.argmax(fused_pred, axis=1)
true_classes = np.argmax(y_test_cat, axis=1)

print(f"\nFused Model Accuracy: {accuracy_score(true_classes, final_classes):.4f}")
print(classification_report(true_classes, final_classes, target_names=target_labels))

os.makedirs("production_models", exist_ok=True)
cnn_model.save("production_models/cnn_model.keras")
ann_model.save("production_models/ann_model.keras")

joblib.dump({
    'scaler': scaler,
    'mean_vals': scaled_selected_means,
    'selected_columns': selected_features 
}, "production_models/tabular_pipeline.pkl")


Data Aligned. Total combined samples: 388

Top 20 Features selected by ANOVA:
['SSS', 'SRS', 'SDS', 'SS%', 'SR%', 'SD%', 'Stress_QPS_Defect', 'Stress_QPS_Extent', 'Stress_QPS_TPD', 'Rest_QPS_Defect', 'Rest_QPS_Extent', 'Rest_QPS_TPD', 'Stress_QGS_SMS', 'Stress_QGS_STS', 'Stress_QGS_SM%', 'Stress_QGS_ST%', 'Stress_QGS_Mot_Ext', 'Stress_QGS_Thk_Ext', 'Rest_QGS_SMS', 'Rest_QGS_Thk_Ext']
Epoch 1/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 35s 2s/step - accuracy: 0.3548 - loss: 2.2957 - val_accuracy: 0.4872 - val_loss: 1.6741 - learning_rate: 2.0000e-04
Epoch 2/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step - accuracy: 0.4581 - loss: 1.9288 - val_accuracy: 0.6282 - val_loss: 1.4967 - learning_rate: 2.0000e-04
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 17s 2s/step - accuracy: 0.6032 - loss: 1.5626 - val_accuracy: 0.6538 - val_loss: 1.4234 - learning_rate: 2.0000e-04
Epoch 4/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step - accuracy: 0.7097 - loss: 1.4004 - val_accuracy: 0.7308 - val_loss: 1.3316 - learning_rate: 2.0000e-

['production_models/tabular_pipeline.pkl']

In [ ]:
import cv2
import easyocr

reader = easyocr.Reader(['en'], gpu=False)


AC_TOP_BOX      = (960,   0, 1280, 180)   
AC_STRESS_BOX   = (960, 180, 1280, 460)   
AC_REST_BOX     = (960, 460, 1280, 720)   

SQGS_STRESS_BOX = (960, 180, 1280, 460)   
SQGS_REST_BOX   = (960, 460, 1280, 720)   

RQGS_STRESS_BOX = (960, 180, 1280, 460)   
RQGS_REST_BOX   = (960, 460, 1280, 720)   


def normalize_text(s):
    if s is None:
        return ""
    s = str(s).strip()
    s = s.replace('|',   'I')
    s = s.replace('—',   '-')
    s = s.replace('cm?', 'cm2')
    s = s.replace('cm°', 'cm2')
    s = s.replace('mI',  'ml')
    return s

def normalize_score_text(s):
    s = s.replace('88S', 'SSS').replace('88s', 'SSS')
    s = s.replace('98S', 'SSS').replace('98s', 'SSS')
    s = s.replace('88%', 'SS%').replace('88 %', 'SS%')
    s = s.replace('S8%', 'SS%').replace('s8 %', 'SS%')
    return s

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def extract_first_number(text):
    text = normalize_text(text)
    m = re.search(r'-?\d+\.?\d*', text)
    return safe_float(m.group()) if m else np.nan

def extract_percent(text):
    text = normalize_text(text)
    m = re.search(r'(\d{1,3})\s*%', text)
    if m:
        return safe_float(m.group(1))
    m = re.search(r'(\d{1,3})\s*9[6b]', text)
    if m:
        return safe_float(m.group(1))
    for match in re.finditer(r'\d+\.?\d*', text):
        val = safe_float(match.group())
        if val <= 100:
            return val
    return np.nan

def extract_cm2(text):
    text = normalize_text(text)
    m = re.search(r'(-?\d+\.?\d*)\s*cm', text, re.IGNORECASE)
    return safe_float(m.group(1)) if m else extract_first_number(text)

def extract_ml(text):
    text = normalize_text(text)
    m = re.search(r'(-?\d+\.?\d*)\s*ml', text, re.IGNORECASE)
    return safe_float(m.group(1)) if m else extract_first_number(text)

def extract_shape_si(text):
    text = normalize_text(text)
    m = re.search(r'(-?\d+\.?\d*)\s*\[\s*SI\s*\]', text, re.IGNORECASE)
    return safe_float(m.group(1)) if m else np.nan

def extract_shape_ecc(text):
    text = normalize_text(text)
    m = re.search(r',\s*(-?\d+\.?\d*)\s*\[\s*Ecc\s*\]', text, re.IGNORECASE)
    return safe_float(m.group(1)) if m else np.nan


def preprocess_for_ocr(pil_img, scale=3):
    img  = np.array(pil_img)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if len(img.shape) == 3 else img.copy()
    h, w = gray.shape
    gray = cv2.resize(gray, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    gray = cv2.filter2D(gray, -1, np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]))
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return thresh

def ocr_region(full_img, box):
    region    = full_img.crop(box)
    processed = preprocess_for_ocr(region, scale=3)
    results   = reader.readtext(processed, detail=0)
    return [str(r).strip() for r in results if str(r).strip()]


KNOWN_KEYS = {
    "Study", "Dataset", "Date", "Database",
    "Volume", "Volure",
    "Area", "Defect", "Extent", "TPD", "Shape",
    "EF", "SMS", "STS", "SM%", "ST%",
    "Mot", "Thk", "Mot Ext", "Thk Ext",
}

def block_lines_to_pairs(lines):
    cleaned = [normalize_text(x) for x in lines if normalize_text(x)]
    pairs = {}
    i = 0
    while i < len(cleaned):
        key = cleaned[i]
        if key in ("Volure", "V0lume", "V0lure"):
            key = "Volume"
        if key == "Mot" and i+1 < len(cleaned) and cleaned[i+1].startswith("Ext"):
            key = "Mot Ext"
            pairs[key] = cleaned[i+2] if i+2 < len(cleaned) else ""
            i += 3; continue
        if key == "Thk" and i+1 < len(cleaned) and cleaned[i+1].startswith("Ext"):
            key = "Thk Ext"
            pairs[key] = cleaned[i+2] if i+2 < len(cleaned) else ""
            i += 3; continue
        if key in KNOWN_KEYS:
            pairs[key] = cleaned[i+1] if i+1 < len(cleaned) else ""
            i += 2
        else:
            i += 1
    return pairs


def parse_top_info(lines):
    data   = {}
    joined = normalize_score_text(" ".join([normalize_text(x) for x in lines]))
    for field in ["SSS", "SRS", "SDS"]:
        m = re.search(rf'{re.escape(field)}\s+(\d+\.?\d*)', joined, re.IGNORECASE)
        if m:
            data[field] = safe_float(m.group(1))
    for field in ["SS%", "SR%", "SD%"]:
        m = re.search(rf'{re.escape(field)}\s*(\d+\.?\d*)', joined, re.IGNORECASE)
        if m:
            data[field] = safe_float(m.group(1))
    sex_m = re.search(r'Sex\s+([A-Z]+)', joined, re.IGNORECASE)
    if sex_m:
        sv = sex_m.group(1).upper()
        data["Sex_Male"]   = 1.0 if sv == "MALE"   else 0.0
        data["Sex_Female"] = 1.0 if sv == "FEMALE" else 0.0
    return data

def parse_qps_block(lines, prefix):
    pairs = block_lines_to_pairs(lines)
    data  = {}
    if "Defect" in pairs: data[f"{prefix}_QPS_Defect"] = extract_cm2(pairs["Defect"])
    if "Extent" in pairs: data[f"{prefix}_QPS_Extent"] = extract_percent(pairs["Extent"])
    if "TPD"    in pairs: data[f"{prefix}_QPS_TPD"]    = extract_percent(pairs["TPD"])
    return data

def parse_qgs_block(lines, prefix):
    pairs = block_lines_to_pairs(lines)
    data  = {}
    vol_key = "Volume" if "Volume" in pairs else ("Volure" if "Volure" in pairs else None)
    if vol_key:            data[f"{prefix}_QGS_Volume"]  = extract_ml(pairs[vol_key])
    if "EF"      in pairs: data[f"{prefix}_QGS_EF"]      = extract_percent(pairs["EF"])
    if "SMS"     in pairs: data[f"{prefix}_QGS_SMS"]     = extract_first_number(pairs["SMS"])
    if "STS"     in pairs: data[f"{prefix}_QGS_STS"]     = extract_first_number(pairs["STS"])
    if "SM%"     in pairs: data[f"{prefix}_QGS_SM%"]     = extract_percent(pairs["SM%"])
    if "ST%"     in pairs: data[f"{prefix}_QGS_ST%"]     = extract_percent(pairs["ST%"])
    if "Mot Ext" in pairs: data[f"{prefix}_QGS_Mot_Ext"] = extract_percent(pairs["Mot Ext"])
    if "Thk Ext" in pairs: data[f"{prefix}_QGS_Thk_Ext"] = extract_percent(pairs["Thk Ext"])
    if "Shape"   in pairs:
        data[f"{prefix}_QGS_SI"]  = extract_shape_si(pairs["Shape"])
        data[f"{prefix}_QGS_Ecc"] = extract_shape_ecc(pairs["Shape"])
    return data


def merge_with_fallback(primary, secondary):
    merged = dict(secondary)
    for k, v in primary.items():
        if k not in merged:
            merged[k] = v
        elif isinstance(v, float) and np.isnan(v):
            pass  
        else:
            merged[k] = v
    return merged


def extract_structured_tabular_data(img_ac_qps, img_stress_qgs, img_rest_qgs):
    """
    Reads each of the 3 images independently using correct crop boxes.

    img_ac_qps     : AC QPS image     → patient scores + QPS stress/rest blocks
    img_stress_qgs : Stress QGS image → QGS stress block (primary) + QGS rest block (secondary)
    img_rest_qgs   : Rest QGS image   → QGS rest block (primary)   + QGS stress block (secondary)
    """
    raw_texts = {
        "ac_top":        ocr_region(img_ac_qps,     AC_TOP_BOX),
        "ac_stress_qps": ocr_region(img_ac_qps,     AC_STRESS_BOX),
        "ac_rest_qps":   ocr_region(img_ac_qps,     AC_REST_BOX),
        "sqgs_stress":   ocr_region(img_stress_qgs, SQGS_STRESS_BOX),
        "sqgs_rest":     ocr_region(img_stress_qgs, SQGS_REST_BOX),
        "rqgs_stress":   ocr_region(img_rest_qgs,   RQGS_STRESS_BOX),
        "rqgs_rest":     ocr_region(img_rest_qgs,   RQGS_REST_BOX),
    }

    top_data   = parse_top_info(raw_texts["ac_top"])
    stress_qps = parse_qps_block(raw_texts["ac_stress_qps"], prefix="Stress")
    rest_qps   = parse_qps_block(raw_texts["ac_rest_qps"],   prefix="Rest")

    stress_qgs = merge_with_fallback(
        primary   = parse_qgs_block(raw_texts["sqgs_stress"], prefix="Stress"),
        secondary = parse_qgs_block(raw_texts["rqgs_stress"], prefix="Stress"),
    )

    rest_qgs = merge_with_fallback(
        primary   = parse_qgs_block(raw_texts["rqgs_rest"],  prefix="Rest"),
        secondary = parse_qgs_block(raw_texts["sqgs_rest"],  prefix="Rest"),
    )

    final_data = {}
    final_data.update(top_data)
    final_data.update(stress_qps)
    final_data.update(rest_qps)
    final_data.update(stress_qgs)
    final_data.update(rest_qgs)

    return final_data, raw_texts

print("Cell 1 loaded — OCR helpers ready.")


In [ ]:
# Grad-CAM helpers 

import tensorflow as tf
import cv2


def verify_gradcam_layer(model):
    """
    Safely finds and prints all conv layers inside the ResNet50 sub-model.
    Works regardless of how Keras exposes output_shape in this version.
    """
    resnet_inner = model.get_layer("resnet50")
    print("Scanning ResNet50 layers for conv targets...")
    conv_layers = []
    for layer in resnet_inner.layers:
        name = layer.name
        # Target conv layers by name pattern — no output_shape needed
        if "conv" in name.lower() and "pad" not in name.lower():
            conv_layers.append(name)

    print(f"Found {len(conv_layers)} conv layers. Last 5:")
    for name in conv_layers[-5:]:
        print(f"  {name}")

    if conv_layers:
        print(f"\n Recommended Grad-CAM target: '{conv_layers[-1]}'")
        return conv_layers[-1]
    else:
        print("No conv layers found — printing ALL layer names for inspection:")
        for layer in resnet_inner.layers:
            print(f"  {layer.name}")
        return None


def get_gradcam_heatmap(model, img_array, last_conv_layer_name="conv5_block3_2_out"):
    """
    Computes a Grad-CAM heatmap for the CNN's top predicted class.

    Parameters
    ----------
    model                : cnn_model  (Keras functional model)
    img_array            : preprocessed 9-channel tensor, shape (1, 224, 224, 9)
    last_conv_layer_name : last conv layer name inside the ResNet50 sub-model.
                           Run verify_gradcam_layer(cnn_model) to get the exact name.

    Returns
    -------
    heatmap        : numpy array (7, 7), values in [0, 1]
    pred_class_idx : int — 0=Normal, 1=Ischemia, 2=Infarction
    """
    resnet_inner = model.get_layer("resnet50")

    # Build helper model: input → [last conv output, final predictions]
    grad_model = tf.keras.models.Model(
        inputs=model.input,
        outputs=[
            resnet_inner.get_layer(last_conv_layer_name).output,
            model.output
        ]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array, training=False)
        pred_class_idx = int(tf.argmax(predictions[0]))
        class_score    = predictions[:, pred_class_idx]

    grads        = tape.gradient(class_score, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap      = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap      = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.math.reduce_max(heatmap) + 1e-8)

    return heatmap.numpy(), pred_class_idx


def overlay_heatmap_on_image(original_pil_img, heatmap, alpha=0.45):
    """
    Overlays a Grad-CAM heatmap on a PIL image.
    Blue = low attention | Red = high attention
    """
    img_w, img_h = original_pil_img.size
    heatmap_resized = cv2.resize(heatmap, (img_w, img_h))
    heatmap_uint8   = np.uint8(255 * heatmap_resized)
    heatmap_colored = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    original_arr    = np.array(original_pil_img.convert("RGB"), dtype=np.float32)
    blended         = (alpha * heatmap_colored.astype(np.float32) + (1 - alpha) * original_arr).clip(0, 255).astype(np.uint8)
    return Image.fromarray(blended)


#  Run verify to find the correct layer name
recommended_layer = verify_gradcam_layer(cnn_model)
print("\nGrad-CAM helpers loaded successfully ")

In [ ]:
import gradio as gr
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input
import tensorflow as tf
import cv2


CNN_MODEL_PATH = 'production_models/cnn_model.keras'
ANN_MODEL_PATH = 'production_models/ann_model.keras
PIPELINE_PATH = 'production_models/tabular_pipeline.pkl'

cnn_model = load_model(CNN_MODEL_PATH)
ann_model = load_model(ANN_MODEL_PATH)
pipeline  = joblib.load(PIPELINE_PATH)

scaler           = pipeline['scaler']
mean_vals        = pipeline['mean_vals']
expected_columns = pipeline['selected_columns']

print("Expected ANN columns:", expected_columns)

#  CNN preprocessing 

CNN_CROP_BOX = (320, 0, 950, 920)

def crop_and_pad(img, target_size=(224, 224)):
    img = img.crop(CNN_CROP_BOX)
    width, height = img.size
    new_width, new_height = target_size
    if width > new_width or height > new_height:
        img.thumbnail((new_width, new_height), Image.LANCZOS)
        width, height = img.size
    pw = (new_width  - width)  // 2
    ph = (new_height - height) // 2
    padded = np.pad(
        np.array(img),
        ((ph, new_height - height - ph), (pw, new_width - width - pw), (0, 0)),
        mode='constant', constant_values=0
    )
    return padded

def preprocess_patient_images(img_ac_qps, img_rest_qgs, img_stress_qgs):
    processed = []
    for img in [img_ac_qps, img_rest_qgs, img_stress_qgs]:
        if img is not None:
            arr = preprocess_input(np.array(crop_and_pad(img.convert('RGB')), dtype=np.float32))
        else:
            arr = preprocess_input(np.zeros((224, 224, 3), dtype=np.float32))
        processed.append(arr)
    return np.expand_dims(np.concatenate(processed, axis=-1), axis=0)

#  ANN input builder 

def build_ann_input_from_ocr(raw_data):
    df = pd.DataFrame([raw_data])
    for col in expected_columns:
        if col not in df.columns:
            df[col] = np.nan
    df       = df[expected_columns]
    X_scaled = scaler.transform(df.values)
    X_scaled = np.nan_to_num(X_scaled, nan=mean_vals)
    return X_scaled, df

#  File to PIL converter 

def file_to_pil(file_obj):
    if file_obj is None:
        return None
    path = file_obj if isinstance(file_obj, str) else file_obj.name
    return Image.open(path).convert("RGB")

#  Grad-CAM (using @tf.function-safe approach) 

def get_gradcam_heatmap(model, img_array):
    """
    Computes Grad-CAM using direct layer access on the loaded model.
    Uses the Conv2D(3,1,1) reducer layer output as a proxy since
    ResNet50 sub-layers may not be traceable after model.load().
    Falls back to a simpler gradient approach if needed.
    """
    img_tensor = tf.cast(img_array, tf.float32)

    # Find the last conv layer that IS directly accessible from cnn_model
    target_layer = None
    for layer in reversed(model.layers):
        if hasattr(layer, 'layers'): 
            for sublayer in reversed(layer.layers):
                if sublayer.name == "conv5_block3_out":
                    target_layer = sublayer
                    break
        if target_layer:
            break

    if target_layer is None:
        print("Warning: conv5_block3_out not found, using fallback")
        return np.zeros((7, 7)), 0

    # Build grad model using the found layer
    try:
        grad_model = tf.keras.models.Model(
            inputs=model.input,
            outputs=[target_layer.output, model.output]
        )

        with tf.GradientTape() as tape:
            conv_outputs, predictions = grad_model(img_tensor, training=False)
            pred_class_idx = int(tf.argmax(predictions[0]))
            class_score    = predictions[:, pred_class_idx]

        grads        = tape.gradient(class_score, conv_outputs)
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
        conv_out     = conv_outputs[0]
        heatmap      = conv_out @ pooled_grads[..., tf.newaxis]
        heatmap      = tf.squeeze(heatmap)
        heatmap      = tf.maximum(heatmap, 0)
        heatmap      = heatmap / (tf.math.reduce_max(heatmap) + 1e-8)

        return heatmap.numpy(), pred_class_idx

    except Exception as e:
        print(f"Grad-CAM grad_model failed: {e}")
        with tf.GradientTape() as tape:
            img_var = tf.Variable(img_tensor)
            predictions = model(img_var, training=False)
            pred_class_idx = int(tf.argmax(predictions[0]))
        return np.zeros((7, 7)), pred_class_idx


def overlay_heatmap_on_image(original_pil_img, heatmap, alpha=0.45):
    img_w, img_h = original_pil_img.size
    heatmap_resized = cv2.resize(heatmap, (img_w, img_h))
    heatmap_uint8   = np.uint8(255 * heatmap_resized)
    heatmap_colored = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    original_arr    = np.array(original_pil_img.convert("RGB"), dtype=np.float32)
    blended         = (alpha * heatmap_colored.astype(np.float32) + (1 - alpha) * original_arr).clip(0, 255).astype(np.uint8)
    return Image.fromarray(blended)

# Predict function 

def predict_patient(file_ac, file_rest, file_stress):
    labels = ['Normal', 'Ischemia', 'Infarction']

    img_ac     = file_to_pil(file_ac)
    img_rest   = file_to_pil(file_rest)
    img_stress = file_to_pil(file_stress)

    if img_ac is None or img_rest is None or img_stress is None:
        return (
            {"Error: Please upload all three required SPECT images.": 1.0},
            "Missing one or more required images.",
            pd.DataFrame(),
            None, None, None
        )

    print(f"Images loaded — AC: {img_ac.size}, Rest: {img_rest.size}, Stress: {img_stress.size}")

    # CNN branch
    X_img = preprocess_patient_images(img_ac, img_rest, img_stress)
    print(f"X_img shape: {X_img.shape}, min: {X_img.min():.2f}, max: {X_img.max():.2f}, mean: {X_img.mean():.2f}")

    # OCR branch
    raw_ocr_data, raw_texts = extract_structured_tabular_data(
        img_ac_qps=img_ac,
        img_stress_qgs=img_stress,
        img_rest_qgs=img_rest,
    )

    missing_cols  = [c for c in expected_columns if c not in raw_ocr_data]
    X_tab, ocr_df = build_ann_input_from_ocr(raw_ocr_data)

    cnn_pred   = cnn_model.predict(X_img, verbose=0)
    ann_pred   = ann_model.predict(X_tab, verbose=0)
    fused_pred = (0.6 * cnn_pred) + (0.4 * ann_pred)

    result = {labels[i]: float(fused_pred[0][i]) for i in range(len(labels))}

    # Grad-CAM
    try:
        heatmap, pred_class_idx = get_gradcam_heatmap(cnn_model, X_img)
        pred_label = labels[pred_class_idx]
        print(f"Grad-CAM done — predicted: {pred_label}, heatmap min: {heatmap.min():.3f}, max: {heatmap.max():.3f}")

        img_ac_display     = img_ac.crop(CNN_CROP_BOX).convert("RGB").resize((224, 224))
        img_rest_display   = img_rest.crop(CNN_CROP_BOX).convert("RGB").resize((224, 224))
        img_stress_display = img_stress.crop(CNN_CROP_BOX).convert("RGB").resize((224, 224))

        cam_ac     = overlay_heatmap_on_image(img_ac_display,     heatmap)
        cam_rest   = overlay_heatmap_on_image(img_rest_display,   heatmap)
        cam_stress = overlay_heatmap_on_image(img_stress_display, heatmap)
    except Exception as e:
        print(f"Grad-CAM failed: {e}")
        cam_ac = cam_rest = cam_stress = None

    debug_text = (
        "=== AC TOP INFO ===\n"           + "\n".join(raw_texts["ac_top"])        +
        "\n\n=== AC STRESS QPS ===\n"     + "\n".join(raw_texts["ac_stress_qps"]) +
        "\n\n=== AC REST QPS ===\n"       + "\n".join(raw_texts["ac_rest_qps"])   +
        "\n\n=== STRESS QGS (from Stress QGS image) ===\n" + "\n".join(raw_texts["sqgs_stress"]) +
        "\n\n"   + "\n".join(raw_texts["sqgs_rest"])   +
        "\n\n=== REST QGS (from Rest QGS image) ===\n"   + "\n".join(raw_texts["rqgs_stress"]) +
        "\n\n"     + "\n".join(raw_texts["rqgs_rest"])   +
        "\n\n=== PARSED FEATURES ===\n"   + "\n".join([f"{k}: {v}" for k, v in raw_ocr_data.items()]) +
        "\n\n=== EXPECTED COLUMNS ===\n"  + "\n".join([str(c) for c in expected_columns]) +
        "\n\n=== MISSING COLUMNS ===\n"   + ("\n".join([str(c) for c in missing_cols]) if missing_cols else "None")
    )

    return result, debug_text, ocr_df, cam_ac, cam_rest, cam_stress


with gr.Blocks(title="MPI Classification Model") as demo:
    gr.Markdown("#  Multimodal Diagnostic Inference")
    gr.Markdown(
        "Upload the 3 required SPECT scans (BMP, PNG, JPG all supported). "
        "Clinical tabular features are extracted automatically from each image's report panel using OCR."
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 1. Image Data")
            file_ac     = gr.File(label="AC QPS (Clear)",     file_types=["image", ".bmp"])
            file_rest   = gr.File(label="Rest QGS (Clear)",   file_types=["image", ".bmp"])
            file_stress = gr.File(label="Stress QGS (Clear)", file_types=["image", ".bmp"])
            submit_btn  = gr.Button("Analyze Patient Data", variant="primary")
        with gr.Column(scale=1):
            gr.Markdown("### 2. Diagnosis Output")
            output_label = gr.Label(num_top_classes=3)

    gr.Markdown("### 3. Grad-CAM Explanations")
    gr.Markdown(
        " **Red/warm** = high CNN attention  |   **Blue/cool** = low attention"
    )
    with gr.Row():
        cam_ac_out     = gr.Image(label="AC QPS — Grad-CAM")
        cam_rest_out   = gr.Image(label="Rest QGS — Grad-CAM")
        cam_stress_out = gr.Image(label="Stress QGS — Grad-CAM")

    gr.Markdown("### 4. OCR Debug Output")
    debug_text = gr.Textbox(lines=35, label="Extracted OCR Text + Parsed Features")

    gr.Markdown("### 5. OCR Features DataFrame")
    output_df = gr.Dataframe(label="OCR Features Sent to ANN")

    submit_btn.click(
        fn=predict_patient,
        inputs=[file_ac, file_rest, file_stress],
        outputs=[output_label, debug_text, output_df,
                 cam_ac_out, cam_rest_out, cam_stress_out]
    )

demo.launch(share=True, show_error=True)
